# Error Analysis: Phishing Detection Classifier

This notebook performs error analysis on the trained phishing detection model:
1. Load trained classifier
2. Compute confusion matrix
3. Identify misclassified phishing samples
4. Plot feature distributions of errors
5. Explain possible causes of misclassification

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.data.dataset import load_processed_dataset, get_feature_names
from src.models.classifier import train_classifier, predict
from src.utils.seed import set_seed

set_seed(42)
sns.set_theme(style="whitegrid")

## 1. Load Data and Train Classifier

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_dataset.csv"

X, y = load_processed_dataset(DATA_PATH)
feature_names = get_feature_names()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {feature_names}")

In [ ]:
model = train_classifier(
    X_train, y_train,
    model_type="xgboost",
    learning_rate=0.01,
    n_estimators=100,
    random_state=42,
)

y_pred = predict(model, X_test)
print("Classifier trained and predictions obtained.")

## 2. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate", "Phishing"],
)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix: Phishing Detection")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (correctly classified legitimate): {tn}")
print(f"False Positives (legitimate → predicted phishing): {fp}")
print(f"False Negatives (phishing → predicted legitimate): {fn}")
print(f"True Positives (correctly classified phishing): {tp}")

## 3. Identify Misclassified Samples

In [ ]:
misclassified_mask = y_test != y_pred
misclassified_indices = np.where(misclassified_mask)[0]

X_mis = X_test[misclassified_indices]
y_mis_true = y_test[misclassified_indices]
y_mis_pred = y_pred[misclassified_indices]

df_mis = pd.DataFrame(X_mis, columns=feature_names)
df_mis["true_label"] = y_mis_true
df_mis["pred_label"] = y_mis_pred
df_mis["error_type"] = np.where(
    y_mis_true == 1, "False Negative (phishing missed)", "False Positive (legitimate flagged)"
)

n_mis = len(df_mis)
n_fn = (df_mis["error_type"] == "False Negative (phishing missed)").sum()
n_fp = (df_mis["error_type"] == "False Positive (legitimate flagged)").sum()

print(f"Total misclassified: {n_mis} / {len(y_test)} ({100 * n_mis / len(y_test):.2f}%)")
print(f"False Negatives (phishing missed): {n_fn}")
print(f"False Positives (legitimate flagged): {n_fp}")
print()
print("Misclassified phishing samples (first 10):")
df_mis[df_mis["true_label"] == 1].head(10)

## 4. Feature Distributions of Errors

In [ ]:
# Build comparison: misclassified vs correctly classified (by error type)
df_test = pd.DataFrame(X_test, columns=feature_names)
df_test["correct"] = ~misclassified_mask
df_test["true_label"] = y_test

# Misclassified phishing (FN) vs correctly classified phishing (TP)
fn_mask = misclassified_mask & (y_test == 1)
tp_mask = ~misclassified_mask & (y_test == 1)
df_fn = df_test[fn_mask][feature_names]
df_tp = df_test[tp_mask][feature_names]

# Misclassified legitimate (FP) vs correctly classified legitimate (TN)
fp_mask = misclassified_mask & (y_test == 0)
tn_mask = ~misclassified_mask & (y_test == 0)
df_fp = df_test[fp_mask][feature_names]
df_tn = df_test[tn_mask][feature_names]

In [ ]:
# Plot: Feature means for FN (missed phishing) vs TP (correctly caught phishing)
if len(df_fn) > 0 and len(df_tp) > 0:
    mean_fn = df_fn.mean()
    mean_tp = df_tp.mean()
    diff = mean_fn - mean_tp
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot: feature means comparison
    x = np.arange(len(feature_names))
    width = 0.35
    axes[0].bar(x - width/2, mean_fn, width, label="FN (missed phishing)", color="coral", alpha=0.8)
    axes[0].bar(x + width/2, mean_tp, width, label="TP (correct phishing)", color="steelblue", alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(feature_names, rotation=45, ha="right")
    axes[0].set_ylabel("Mean feature value")
    axes[0].set_title("Feature Means: Missed Phishing vs Correctly Caught Phishing")
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)
    
    # Difference (FN - TP)
    colors = ["coral" if d > 0 else "steelblue" for d in diff]
    axes[1].barh(feature_names, diff, color=colors, alpha=0.8)
    axes[1].axvline(0, color="black", linewidth=0.5)
    axes[1].set_xlabel("Mean(FN) - Mean(TP)")
    axes[1].set_title("Feature difference: Missed phishing has higher values where bar is positive")
    axes[1].grid(axis="x", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot: Feature means for FP (false flagged legitimate) vs TN (correct legitimate)
if len(df_fp) > 0 and len(df_tn) > 0:
    mean_fp = df_fp.mean()
    mean_tn = df_tn.mean()
    diff = mean_fp - mean_tn
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(len(feature_names))
    width = 0.35
    axes[0].bar(x - width/2, mean_fp, width, label="FP (false flagged legitimate)", color="coral", alpha=0.8)
    axes[0].bar(x + width/2, mean_tn, width, label="TN (correct legitimate)", color="steelblue", alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(feature_names, rotation=45, ha="right")
    axes[0].set_ylabel("Mean feature value")
    axes[0].set_title("Feature Means: False Positives vs Correctly Classified Legitimate")
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)
    
    colors = ["coral" if d > 0 else "steelblue" for d in diff]
    axes[1].barh(feature_names, diff, color=colors, alpha=0.8)
    axes[1].axvline(0, color="black", linewidth=0.5)
    axes[1].set_xlabel("Mean(FP) - Mean(TN)")
    axes[1].set_title("Feature difference: False positives look more 'phishing-like' where bar is positive")
    axes[1].grid(axis="x", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap: feature distributions for misclassified samples by error type
if len(df_mis) > 0:
    df_plot = df_mis.groupby("error_type")[feature_names].mean()
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df_plot.T, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0.5, ax=ax)
    ax.set_title("Mean Feature Values by Error Type")
    ax.set_xlabel("Error type")
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution of top discriminative features for misclassified phishing (FN)
if len(df_fn) > 0:
    top_features = ["URL_Length", "URL_Depth", "iFrame", "Mouse_Over", "Right_Click", "Web_Forwards"]
    top_features = [f for f in top_features if f in feature_names][:6]
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, feat in enumerate(top_features):
        ax = axes[i]
        ax.hist(df_tp[feat] if len(df_tp) > 0 else [], bins=10, alpha=0.6, label="TP (correct)", color="steelblue", density=True)
        ax.hist(df_fn[feat], bins=10, alpha=0.6, label="FN (missed)", color="coral", density=True)
        ax.set_title(feat)
        ax.set_xlabel("Value")
        ax.legend()
    plt.suptitle("Feature distributions: Missed phishing (FN) vs Correctly caught phishing (TP)")
    plt.tight_layout()
    plt.show()

## 5. Possible Causes of Misclassification

### False Negatives (Phishing Missed)

Phishing samples incorrectly classified as legitimate often exhibit:

1. **Legitimate-like feature profiles**: Missed phishing URLs may have shorter URLs, valid DNS records, fewer suspicious HTML/JS indicators (no iFrame, no right-click disable, no web forwarding).

2. **Sophisticated mimicry**: Attackers increasingly mimic legitimate sites by using HTTPS, proper domain structure, and avoiding obvious red flags.

3. **Feature overlap**: Some phishing sites share feature distributions with legitimate sites (e.g., new but legitimate startups vs phishing).

4. **Class imbalance or scaling**: If the phishing class is underrepresented in certain feature regions, the model may under-predict phishing.

### False Positives (Legitimate Flagged)

Legitimate samples incorrectly flagged as phishing often exhibit:

1. **Phishing-like features**: Legitimate sites may use URL shorteners, have deep paths, use iframes for ads, or disable right-click for security—features that overlap with phishing.

2. **New or atypical domains**: Startups, personal sites, or regional sites may have young domains, unusual structures, or minimal DNS history.

3. **Feature noise**: Binary/categorical features can be noisy; a single "suspicious" feature may push the model toward phishing.

### Recommendations

- **GAN augmentation**: Add synthetic phishing samples to expose the model to edge cases.
- **Feature engineering**: Consider additional features (e.g., domain reputation, lexical patterns).
- **Threshold tuning**: Adjust decision threshold to trade off recall vs precision based on use case.
- **Ensemble methods**: Combine multiple classifiers to reduce variance.